# 01 · Node set

**Owner:** Avneesh  
**Reads:** openFDA adverse-event reports  
**Writes:** `data/processed/nodes.csv`

The ~300 most frequently reported single-ingredient drugs, mapped to RxNorm ingredients. Both layers use this node set.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
REFERENCE = ROOT / "data" / "reference"
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"

import os

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
API_KEY = os.getenv("OPENFDA_API_KEY", "")
print("API key found" if API_KEY else "No API key: add it to .env (see .env.example)")

API key found


In [2]:
import sys

sys.path.insert(0, str(ROOT / "notebooks"))
import openfda_utils as fu

In [3]:
N_DRUGS = 300        # drugs in the network
N_CANDIDATES = 1000  # most-reported substance names to consider (openFDA's maximum per count query)
RXNAV_THREADS = 8    # parallel RxNorm lookups (RxNav allows 20 requests/second)

## Step 1 · Candidate names

The most frequently reported substance names in the adverse-event database. One drug can appear under several names (salts such as *warfarin sodium*), and some names are vitamins or combination products; Step 2 sorts this out.

In [10]:
resp = fu.openfda(RAW, API_KEY, "event", {"count": fu.SUB, "limit": N_CANDIDATES}, "top_substances")
candidates = pd.DataFrame(resp["results"]).rename(columns={"term": "fda_name", "count": "name_count"})
print(f"{len(candidates)} candidate names; event data last updated {resp['meta']['last_updated']}")
candidates.head(10)

1000 candidate names; event data last updated 2026-10-28


,fda_name,name_count
0,ACETAMINOPHEN,615132
1,ETANERCEPT,597173
2,ASPIRIN,568825
3,DUPILUMAB,518489
4,IBUPROFEN,516893
5,PREDNISONE,508041
6,METHOTREXATE SODIUM,492307
7,METFORMIN HYDROCHLORIDE,445559
8,OMEPRAZOLE MAGNESIUM,443443
9,LENALIDOMIDE,425916


## Step 2 · Map every name to its RxNorm ingredient

For each name: RxNorm code of the name → its active ingredient(s).

* one ingredient → `single` (kept)
* several ingredients → `combination` (dropped: a combination product is not one drug)
* no RxNorm match → `not_found` (dropped; listed in the checks below)
* vitamins, minerals, herbal products, electrolytes → `non_drug` (dropped)

In [5]:
from concurrent.futures import ThreadPoolExecutor

def map_name(name):
    status, ins = fu.rxnorm_ingredients(RAW, name)
    if status == "single" and fu.NON_DRUG.search(ins[0][1]):
        status = "non_drug"
    if status == "single" or status == "non_drug":
        return {"fda_name": name, "status": status, "rxcui": ins[0][0], "ingredient": ins[0][1]}
    return {"fda_name": name, "status": status, "rxcui": None,
            "ingredient": " + ".join(n for _, n in ins) or None}

with ThreadPoolExecutor(RXNAV_THREADS) as pool:
    mapped = list(pool.map(map_name, candidates["fda_name"]))

mapping = candidates.merge(pd.DataFrame(mapped), on="fda_name")
mapping.to_csv(PROCESSED / "name_mapping.csv", index=False)
mapping["status"].value_counts()

status
single      851
non_drug    149
Name: count, dtype: int64

## Step 3 · One row per ingredient, top drugs first

Names that map to the same ingredient are merged; `fda_names` keeps every name, because the openFDA queries in notebooks 02 and 03 search for all of them. The ranking count adds up the names' counts, so a report listing two names of one drug is counted twice; Step 4 replaces it with an exact count.

In [6]:
single = mapping[mapping["status"] == "single"]
by_ingredient = (single.groupby(["rxcui", "ingredient"])
                 .agg(fda_names=("fda_name", lambda s: "|".join(sorted(s))),
                      rank_count=("name_count", "sum"))
                 .reset_index()
                 .sort_values("rank_count", ascending=False))
chosen = by_ingredient.head(N_DRUGS).reset_index(drop=True)
print(f"{len(by_ingredient)} distinct ingredients; keeping the top {len(chosen)}")

730 distinct ingredients; keeping the top 300


## Step 4 · Exact report counts

`n_reports` = number of reports that list the drug under any of its names (one query per drug).

In [7]:
def n_reports(names):
    q = fu.or_query(fu.SUB, names.split("|"))
    return fu.total(fu.openfda(RAW, API_KEY, "event", {"search": q, "limit": 1}, f"N_{names[:40]}"))

chosen["n_reports"] = [n_reports(n) for n in chosen["fda_names"]]
nodes = (chosen.sort_values("n_reports", ascending=False)
         .reset_index(drop=True)
         .rename_axis("node_id").reset_index()
         [["node_id", "rxcui", "ingredient", "n_reports", "fda_names"]])
nodes.to_csv(PROCESSED / "nodes.csv", index=False)
nodes.head(20)

,node_id,rxcui,ingredient,n_reports,fda_names
0,0,161,acetaminophen,615132,ACETAMINOPHEN
1,1,214555,etanercept,597173,ETANERCEPT
2,2,1191,aspirin,568825,ASPIRIN
3,3,1876376,dupilumab,518489,DUPILUMAB
4,4,5640,ibuprofen,517140,IBUPROFEN|IBUPROFEN SODIUM
5,5,8640,prednisone,508041,PREDNISONE
6,6,6851,methotrexate,494382,METHOTREXATE|METHOTREXATE SODIUM
7,7,6809,metformin,445559,METFORMIN HYDROCHLORIDE
8,8,7646,omeprazole,444229,OMEPRAZOLE|OMEPRAZOLE MAGNESIUM
9,9,17767,amlodipine,434733,AMLODIPINE|AMLODIPINE BENZOATE|AMLODIPINE BESY...


## Checks

Look through these before moving on:

1. no ingredient appears twice;
2. no well-known drug is among the `not_found` names (if one is, add its spelling to the mapping by hand);
3. the `combination` names really are combination products.

In [8]:
assert nodes["rxcui"].is_unique and nodes["ingredient"].is_unique
print(f"nodes: {len(nodes)}, reports per drug from {nodes.n_reports.min():,} to {nodes.n_reports.max():,}")
print("\nMost-reported names that RxNorm did not recognize:")
print(mapping[mapping.status == "not_found"].nlargest(15, "name_count")[["fda_name", "name_count"]].to_string(index=False))
print("\nMost-reported combination products (dropped):")
print(mapping[mapping.status == "combination"].nlargest(10, "name_count")[["fda_name", "ingredient"]].to_string(index=False))
print("\nAPI requests:", fu.STATS)

nodes: 300, reports per drug from 17,611 to 615,132

Most-reported names that RxNorm did not recognize:
Empty DataFrame
Columns: [fda_name, name_count]
Index: []

Most-reported combination products (dropped):
Empty DataFrame
Columns: [fda_name, ingredient]
Index: []

API requests: {'requests': 25, 'cached': 2641}
